# 00 Dataset Sampling

Samples a balanced 250-image subset from NIH ChestX-ray14 (50 images per pathology: Cardiomegaly, Effusion, Pneumonia, Pneumothorax, Atelectasis), restricted to pure single-label cases to avoid comorbidity confounds, then locates and copies the matching image files from the extracted NIH archives into `data/`.

**Prerequisite:** the 12 NIH ChestX-ray14 zip archives must already be downloaded and extracted somewhere on disk (see [the official source](https://nihcc.app.box.com/v/ChestXray-NIHCC)) before running this notebook - no manifest exists mapping filenames to archives, so this step cannot be automated further.

In [ ]:
import pandas as pd
import src.config as config

In [ ]:
# Load the full NIH label metadata (`Data_Entry_2017_v2020.csv`), covering all ~112,000 images.
dataset_csv = pd.read_csv('../data/Data_Entry_2017_v2020.csv')
print(dataset_csv.columns)
dataset_csv.head(10)

In [ ]:
# Inspect label frequency for Cardiomegaly, confirming the pure single-label counts available for the target pathology before sampling
condition = dataset_csv['Finding Labels'] == 'Cardiomegaly'
cardiomegaly_only = dataset_csv[condition]
cardiomegaly_only.sample(n=50, random_state=config.SEED)
len(cardiomegaly_only)

1093

In [ ]:
# For each of the five target pathologies, filter to pure single-label rows and sample 50 with a fixed seed, then combine into one balanced 250-row dataframe
sampled_dataset = []
for pathology in config.PATHOLOGIES:
    condition = dataset_csv['Finding Labels'] == pathology
    filtered_csv = dataset_csv[condition]
    print(pathology, " ", len(filtered_csv))
    filtered_csv = filtered_csv.sample(n=50, random_state=config.SEED)
    sampled_dataset.append(filtered_csv)

final_dataset = pd.concat(sampled_dataset)
print(len(final_dataset))

print(final_dataset['Finding Labels'].value_counts())
final_dataset = final_dataset[['Image Index', 'Finding Labels']]

# Save the sampled 250-row label set to `data/sampled_data_labels.csv` for reuse later
print(final_dataset.shape)
print(final_dataset.head(5))
final_dataset.to_csv('../data/sampled_data_labels.csv', index=False)

In [ ]:
from pathlib import Path
import shutil

# Search the extracted NIH archive folders for the 250 sampled filenames and copy each match into `data/`, reporting any filenames not found
def find_and_copy_images(target_filenames: list[str], source_root: Path, dest_dir: Path) -> list[str]:
    dest_dir.mkdir(parents=True, exist_ok=True)
    target_set = set(target_filenames)
    found = set()

    for image_path in source_root.rglob("*.png"):
        if image_path.name in target_set:
            shutil.copy2(image_path, dest_dir / image_path.name)
            found.add(image_path.name)

    missing = list(target_set - found)
    return missing

In [ ]:
# Confirm all 250 target images were located and copied successfully
target_filenames = final_dataset["Image Index"].tolist()

missing = find_and_copy_images(
    target_filenames=target_filenames,
    source_root=Path("D:/xrays"),  # edit to wherever you extracted the NIH archives
    dest_dir=Path("../data/xray_dataset"),
)

print(f"Copied {len(target_filenames) - len(missing)} / {len(target_filenames)} images.")